# T81-577 Applied Data Science Individual Project
# Mobile Strategy Games User Rating Prediction
# Tong Yan

## Table of Contents:

- Project Overview


- Data Exploration


- Data Preprocessing


- Predictive Modelling

  4.1 Split dataset into train/test

  4.2 Create pipelines

  4.3 Model fitting & Making prediction
  
        4.3.1 Linear Regression

        4.3.2 Random Forest

        4.3.3 Gradient Boosting

- Model Comparasion & Selection


- Conclusion


- Business applications & limitations
    

## 1. Project Overview

For this project, I will be using the 'appstore_games.csv' dataset (https://www.kaggle.com/tristan581/17k-apple-app-store-strategy-games) to predict the average user ratings of mobile strategy games using features including game genres, app price, in-app purchases, app size, etc. Game descriptions and text mining will not be involved in this project. 

I will primarily focus on data exploration, data preprocessing and predictive modelling. I will try 3 different regression models, and use Hyperparameter Tuning methods to try to improve model performance. Then I will decide a final model with the lowest ROOT MEAN SQURE ERROR (RMSE) and output the predicting result for the testing dataset. 

## 2. Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

help(sns)

In [ ]:
# Have a primary look at the dataset
df = pd.read_csv('https://raw.githubusercontent.com/stellayannn/DataScience_TongYan/master/data/appstore_games.csv', na_values = ['NA', '?'])
df.info()

In [ ]:
# Look at the summary of null value
df.isnull().sum()

In [ ]:
# Get familiar with the dataframe structure
df.head(5)

In [ ]:
# Drop useless columns
df = df.drop(['URL','ID','Name','Icon URL', 'Primary Genre', 'Subtitle', 
              'Description', 'Developer', 'Languages','Original Release Date',
              'Current Version Release Date'], axis = 1)

# Rename columns
df.columns = ['ave_rating', 'rating_count', 'price', 'inapp_purchase', 
              'age_rating', 'size', 'genre']

In [ ]:
df.head(10)

In [ ]:
# Have a look at the distribuiton of average rating
plt.figure(figsize = (10,6))
img1 = sns.countplot(x = df['ave_rating'])
img1.set(xlabel = 'average rating', ylabel = 'count')
img1.plot()

In [ ]:
# Have a look at the distribuiton of price
plt.figure(figsize = (20,10))
img2 = sns.countplot(x = df['price'])
img2.set(xlabel = 'price', ylabel = 'count')
img2.plot()

In [ ]:
# Have a look at the distribuiton of age ratings
plt.figure(figsize = (10,6))
img3 = sns.countplot(x = df['age_rating'])
img3.set(xlabel = 'age rating', ylabel = 'count')
img3.plot()

In [ ]:
# Create a list of genres
list = []
for i in range(len(df)):
    for j in df['genre'][i].split(','):
        list.append(j)

# Select the top20 most common genres from the list
from collections import Counter
c = Counter(list)
top_genre = pd.DataFrame(c.most_common(20))

In [ ]:
# Have a look at the distribuiton of gneres
plt.figure(figsize = (25,10))
img4 = sns.barplot(x = top_genre[0], y = top_genre[1])
img4.set(xlabel = 'genre', ylabel = 'count')
img4.plot()

In [ ]:
# Have a look at the distribuiton of app sizes
plt.figure(figsize = (16,8))
img5 = sns.distplot(df['size'])
img5.set(xlabel = 'size', ylabel = 'count')
img5.plot()

## 3. Data Preprocessing

In [ ]:
# Drop the rows wihtout 'ave_rating' value
df1 = df.drop(df[np.isnan(df['ave_rating'])].index)

In [ ]:
# Drop the rows wihtout 'size' value
df1.drop(df1[np.isnan(df1['size'])].index, inplace = True)

# Drop the rows with 'rating_count' less than 5
df1.drop(df1[df1['rating_count'] < 5].index, inplace = True)

df1.reset_index(inplace = True, drop = True)

In [ ]:
# Impute values for column 'inapp_purchase' 
df1['inapp_purchase'].fillna(int(0), inplace = True)
df1

In [ ]:
# Replace objects in 'inapp_purchase' with mean of in-app purchase prices of that game
for i in range(len(df1)): 
    # Assign non-null object to a variable and replace the cell with 0
    if df1.loc[i, 'inapp_purchase'] != 0:
        inapp = df1.loc[i, 'inapp_purchase'].split(',')
        df1.loc[i, 'inapp_purchase'] = 0
        # Calculate mean of values and assign back to dataframe
        c = 0
        for j in inapp:
            df1.loc[i, 'inapp_purchase'] = float(df1.loc[i, 'inapp_purchase']) + float(j)
            c += 1
        df1.loc[i, 'inapp_purchase'] = df1.loc[i, 'inapp_purchase']/c

# Convert to numeric variable
df1['inapp_purchase'] = pd.to_numeric(df1['inapp_purchase'])

In [ ]:
# Remove '+' in 'age_rating' and convert to integer
for i in range(len(df1)):
    for j in (4,9,12,17):
        if ('{}+'.format(j)) == str(df1.loc[i, 'age_rating']):
            df1.loc[i, 'age_rating'] = j

# Convert to numeric variable
df1['age_rating'] = pd.to_numeric(df1['age_rating'])

In [ ]:
# Convert 'genre' to dummy variables
genre = np.unique(list).tolist()

# Remove space before some genres
genres = []
for i in range(len(genre)):
    genres.append(genre[i].strip())
genres

In [ ]:
# Convert 'genre' to dummy variables
# As there are multiple elements in one cell, I convert 'genre' to dummy variables manually
# Create columns for dummy variables and assign 0s
for g in genres:
    df1['genre_{}'.format(g)] = 0

In [ ]:
# Change cells to 1 if corresponding genres appear in same row
for i in genres:
    for j in range(len(df1)):
        if i in df1.loc[j, 'genre']:
            df1.loc[j, 'genre_{}'.format(i)] = 1

# Drop the original 'genre' column
df1.drop(['genre'], axis = 1, inplace = True)
df1

## 4. Predictive Modelling

### 4.1 Split dataset into train/test

In [ ]:
from sklearn.model_selection import train_test_split

# Split the datasets into training and testing data
df_x = df1.drop(['ave_rating'], axis = 1)
X = df_x
y = df1['ave_rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 5)

### 4.2 Create pipelines

In [ ]:
# Create a pipeline of imputing empty values, creating polynomial features and feature scaling
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Create a transformer
transformer = ColumnTransformer([("norm1", SimpleImputer(missing_values = np.nan, strategy = 'mean'),[2]),
                                 ('poly', PolynomialFeatures(3),[2,3])])

# Create pipelines of three different regression algorithms
# Linear Regression pipeline
ln_model = Pipeline(steps=([('transformer', transformer),
                            ('scaler',MinMaxScaler()),
                            ('LR1', LinearRegression())]))

# Random Forest pipeline
rf_model = Pipeline(steps=([('transformer', transformer),
                            ('scaler',MinMaxScaler()),
                            ('LR2', RandomForestRegressor())]))

# Gradient Boosting pipeline
gb_model = Pipeline(steps=([('transformer', transformer),
                            ('scaler',MinMaxScaler()),
                            ('LR3', GradientBoostingRegressor())]))

In [ ]:
# Verify the number of missing value
df1.isnull().sum().sum()

### 4.3 Model fitting & Making prediction

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn import metrics

# Create an empty dataframe for RMSE scores
df_rmse = pd.DataFrame(columns = ['model', 'RMSE'])

In [ ]:
# Define a function to evaluate and record model accuracy
def rmse_score(pred, model_name):
    """Print out RMSE score of a model and add the score to dataframe"""
    rmse = metrics.mean_squared_error(y_test, pred)
    to_append = [model_name, rmse]
    df_rmse.loc[len(df_rmse)] = to_append
    return ('RMSE for this model is: {}'.format(rmse))

### 4.3.1 Linear Regression

In [ ]:
# Have a look at the parameters
LinearRegression()

In [ ]:
# Fit linear model into the training data and make prediction
ln = ln_model.fit(X_train,y_train)
y_pred1 = ln_model.predict(X_test)

In [ ]:
# Call the rmse_score function to get rmse score
rmse_score(y_pred1, 'Linear')

### 4.3.2  Random Forest

In [ ]:
# Have a look at the parameters
RandomForestRegressor()

In [ ]:
# Fit Random Forest model into the training data and make prediction
rf = rf_model.fit(X_train,y_train)
y_pred2 = rf_model.predict(X_test)

In [ ]:
# Call the rmse_score function to get rmse score
rmse_score(y_pred2, 'RandomForest')

#### Grid Search

In [ ]:
# Try Grid Search for hyperparameter tuning

# Set up a dictionary with hyperparameters to test
param_grid = {
               'LR2__min_samples_split': [1, 2, 3],
               'LR2__min_samples_leaf': [1, 2, 3],
               'LR2__n_estimators': [20, 50, 100, 200, 300]
}

# Initialize Grid Rearch and fit data into the model
grid_search = GridSearchCV(estimator = rf_model, param_grid = param_grid, cv = 3, n_jobs = -1, verbose = 2)
grid_search.fit(X_train, y_train)

# Check out the parameters that return the highest accuracy
best_grid = grid_search.best_estimator_
best_grid

In [ ]:
# Make prediction
y_pred3 = best_grid.predict(X_test)

# Call the rmse_score function to get rmse score
rmse_score(y_pred3, 'RandomForest_grid')

#### Randomized Search

In [ ]:
# Try Randomized Search for hyperparameter tuning

# Set up a dictionary with hyperparameters to test
param_dist = {
               'LR2__min_samples_split': np.linspace(1, 10, 8, dtype = int),
               'LR2__min_samples_leaf': np.linspace(1, 10, 8, dtype = int),
               'LR2__n_estimators': np.linspace(20, 500, 10, dtype = int)}

# Initialize Randomized Rearch and fit data into the model
random_search = RandomizedSearchCV(estimator = rf_model, param_distributions = param_dist, 
                                   cv = 3, n_jobs = -1, verbose = 2,n_iter = 10)
random_search.fit(X_train, y_train)

# Check out the parameters that return the highest accuracy
best_random= random_search.best_estimator_
best_random

In [ ]:
# Make prediction
y_pred4 = best_random.predict(X_test)

# Call the rmse_score function to get rmse score
rmse_score(y_pred4, 'RandomForest_randomized')

### 4.3.3 Gradient Boosting Regression

In [ ]:
# Have a look at the parameters
GradientBoostingRegressor()

In [ ]:
# Fit Gradient Boosting model into the training data and make prediction
gb = gb_model.fit(X_train,y_train)
y_pred5 = gb_model.predict(X_test)

In [ ]:
# Make prediction
y_pred5 = gb_model.predict(X_test)

# Call the rmse_score function to get rmse score
rmse_score(y_pred5, 'GradientBoosting')

#### Grid Search

In [ ]:
# Try Grid Search for hyperparameter tuning

# Set up a dictionary with hyperparameters to test
param_grid = {
               'LR3__max_depth': [2, 3, 4, 5],
               'LR3__learning_rate': [0.03, 0.05, 0.1, 0.15, 0.2],
               'LR3__n_estimators': [30, 50, 100, 200, 400],
               'LR3__min_samples_split': [1, 2, 3]
}

# Initialize Grid Rearch and fit data into the model
grid_search = GridSearchCV(estimator = gb_model, param_grid = param_grid, cv = 3, n_jobs = -1, verbose = 2)
grid_search.fit(X_train, y_train)

# Check out the parameters that return the highest accuracy
best_grid = grid_search.best_estimator_
best_grid

In [ ]:
# Make prediction
y_pred6 = best_grid.predict(X_test)

# Call the rmse_score function to get rmse score
rmse_score(y_pred6, 'GradientBoosting_grid')

#### Randomized Search

In [ ]:
# Try Randomized Search for hyperparameter tuning

# Set up a dictionary with hyperparameters to test
param_dist = {
               'LR3__max_depth': np.linspace(1, 10, 5, dtype = int),
               'LR3__learning_rate': np.linspace(0.03, 0.95, 8, dtype = float),
               'LR3__n_estimators': np.linspace(30, 600, 8, dtype = int),
               'LR3__min_samples_split': np.linspace(1, 5, 3, dtype = int)}

# Initialize Randomized Rearch and fit data into the model
random_search = RandomizedSearchCV(estimator = gb_model, param_distributions = param_dist, 
                                   cv = 3, n_jobs = -1, verbose = 2,n_iter = 10)
random_search.fit(X_train, y_train)

# Check out the parameters that return the highest accuracy
best_random= random_search.best_estimator_
best_random

In [ ]:
# Make prediction
y_pred7 = best_random.predict(X_test)

# Call the rmse_score function to get rmse score
rmse_score(y_pred7, 'GradientBoosting_randomized')

## 5. Model Comparasion & Selection

In [ ]:
# Sort df_rmse in acsending order and display the dataframe
df_rmse.sort_values(['RMSE'], ascending = True, inplace = True)
df_rmse.reset_index(inplace = True, drop = True)
display(df_rmse)

In [ ]:
# Plot the dataframe in a line chart
df_rmse.plot(x = 'model',
             y = 'RMSE',
             title = 'RMSE of models')
plt.xticks(range(len(df_rmse['model'])),
           df_rmse['model'],
           fontsize = 9,
           rotation = 35)
plt.show()

## 6. Conclusion

As we can see from the above results, although the differences in RMSE scores are not obvious, Gradient Boosting Methods have best predictions with lowest RMSE scores. Linear Regression performs better than basic Random Forest, which is beyond my expectations. The overall accuracy of models on this dataset is: Gradient Boosting > Linear > Random Forest.

As for Hyperparameter Tuning methods, I tried Grid Search and Randomized Search. The result suggests that Randomized Research performs better on both Gradient Boosting and Random Forest model (Linear model does not need a hyperparameter tuning). Thus, the effectiveness of Hyperparameter Tuning methods here is: Randomized Research > Grid Research. This result could be influenced by the parameters I choose. 

Thus, I choose the Gradient Boosting Model with Randomized Research hyperparameter tuning method as my final model to make prediction. 

In [ ]:
# Create a dataframe of the final testing prediction
df_test_result = pd.DataFrame(y_pred7, columns = ['pred'])
# Export the dataframe to a csv file
df_test_result.to_csv('game_pred.csv', index = False)
df_test_result

## 7. Business applications & limitations

This program can be used to predict the average user ratings of mobile stratrgy games, and provide some useful instructions to mobile strategy game developers on the features of a more popular mobile game. App developers can use this model to predict whether their product would be rated a higher or lower score (only for reference).

However, the model has a lot of limitations. Firstly, the largest limitaiton is from the dataset itself. There are only limited number of features that can be used to train the model, many of which do not have strong connections to the target variable. This implies that its reference value is very limited. Besides, I dropped all the columns with descriptive words since I did not intend to include any text mining technique in my model. What I did is to convert 'genres' into dummy variables so that the model can be more persuasive and meaningful. If the 'Name', 'Subtitle' and 'Description' features are explored using text mining, the result might be much better. Since text mining is not covered in this course, I may try this in the future. Moreover, I only tried 3 different regression models and used limited number of hyperparameters in order to control the total runtime. I think the results could be slightly better if I have more time to try different parameter combos and models. 

Thank you!

In [ ]:
# pip freeze > requirement.txt